In [1]:
pip install requests pandas numpy sentence-transformers pyarrow tqdm


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import requests
import pandas as pd
import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# --- Configuration ---
# Token lives in .env (gitignored): TMDB_API_KEY=...
for line in Path(".env").read_text().splitlines() if Path(".env").exists() else []:
    key, _, value = line.partition("=")
    os.environ.setdefault(key.strip(), value.strip())
TMDB_API_KEY = os.environ["TMDB_API_KEY"]
HEADERS = {
    "accept": "application/json",
    "Authorization": f"Bearer {TMDB_API_KEY}"
}
PAGES = 300                       # 20 films per page; TMDb caps top_rated at 500 pages
MAX_WORKERS = 10                  # parallel requests; TMDb allows roughly 40-50 req/s
MODEL_NAME = "all-mpnet-base-v2"  # 768-D, noticeably better than all-MiniLM-L6-v2
OUT_DIR = Path(".")

session = requests.Session()
session.headers.update(HEADERS)


def get_json(url, retries=5):
    """GET with backoff on rate limiting (429) and transient errors."""
    for attempt in range(retries):
        try:
            response = session.get(url, timeout=10)
        except requests.RequestException:
            time.sleep(2 ** attempt)
            continue
        if response.status_code == 200:
            return response.json()
        if response.status_code == 429 or response.status_code >= 500:
            time.sleep(float(response.headers.get("Retry-After", 2 ** attempt)))
            continue
        return None  # 404 etc. - not worth retrying
    return None


def fetch_parallel(urls, desc):
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = [pool.submit(get_json, url) for url in urls]
        results = [f.result() for f in tqdm(as_completed(futures), total=len(futures), desc=desc)]
    return [r for r in results if r]


# 1. Fetch top-rated movie IDs using TMDb's official top_rated endpoint
def fetch_top_movie_ids(pages):
    urls = [f"https://api.themoviedb.org/3/movie/top_rated?language=en-US&page={p}" for p in range(1, pages + 1)]
    pages_data = fetch_parallel(urls, "Fetching Movie Pages")
    # Ranks can shift while paging, so de-duplicate
    return list({movie["id"] for page in pages_data for movie in page.get("results", [])})


# 2. Fetch full details for each film
def fetch_movie_details(movie_ids):
    # append_to_response bundles the keywords into the same request
    urls = [
        f"https://api.themoviedb.org/3/movie/{movie_id}?language=en-US&append_to_response=keywords"
        for movie_id in movie_ids
    ]
    details = fetch_parallel(urls, "Fetching Movie Details")
    return pd.DataFrame(
        [
            {
                "id": d.get("id"),
                "title": d.get("title", ""),
                "tagline": d.get("tagline") or "",
                "genres": ", ".join(g["name"] for g in d.get("genres", [])),
                "keywords": ", ".join(k["name"] for k in d.get("keywords", {}).get("keywords", [])),
                "overview": d.get("overview") or "",
                "release_date": d.get("release_date") or "",
                "vote_average": d.get("vote_average", 0.0),
            }
            for d in details
        ],
        columns=["id", "title", "tagline", "genres", "keywords", "overview", "release_date", "vote_average"],
    )


# --- Execution Pipeline ---

if __name__ == "__main__":
    print("--- Step 1: Gathering TMDb Data ---")
    ids = fetch_top_movie_ids(PAGES)
    print(f"Found {len(ids)} unique films.")

    if not ids:
        print("\n❌ Error: No movie IDs were returned. Please check your TMDb API key/token.")
    else:
        df = fetch_movie_details(ids)
        # Films with no overview have nothing meaningful to embed
        df = df[df["overview"].str.strip() != ""].sort_values("id").reset_index(drop=True)

        print("\n--- Step 2: Building Text To Embed ---")
        # Title and tagline are deliberately left out: titles made films match on shared
        # words ("The Bad Guys" ~ "The Good, the Bad and the Ugly"), and taglines are
        # wordplay ("Your mind is the scene of the crime") that reads as literal content.
        # Keywords capture what a film is about when the overview is thin.
        df["concat_text"] = (
            "Genres: " + df["genres"] + ". " +
            "Keywords: " + df["keywords"] + ". " +
            "Overview: " + df["overview"]
        )
        print(f"Prepared {len(df)} movies for embedding.")

        print(f"\n--- Step 3: Generating Embeddings ({MODEL_NAME}) ---")
        model = SentenceTransformer(MODEL_NAME)
        embeddings = model.encode(
            df["concat_text"].tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=True
        )

        print("\n--- Step 4: Saving Files ---")
        np.save(OUT_DIR / "movie_embeddings.npy", embeddings)
        df.drop(columns="concat_text").to_parquet(OUT_DIR / "movie_metadata.parquet")

        print(f"\n✅ Complete! Saved {embeddings.shape[0]} embeddings of dimension {embeddings.shape[1]}.")


W1019 22:55:58.113000 48293 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


--- Step 1: Gathering TMDb Data ---


Fetching Movie Pages: 100%|██████████████████| 300/300 [00:01<00:00, 177.42it/s]


Found 5963 unique films.


Fetching Movie Details: 100%|███████████████| 5963/5963 [03:16<00:00, 30.35it/s]



--- Step 2: Building Text To Embed ---
Prepared 5963 movies for embedding.

--- Step 3: Generating Embeddings (all-mpnet-base-v2) ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]


--- Step 4: Saving Files ---

✅ Complete! Saved 5963 embeddings of dimension 768.
